## Notes

In [ ]:
# can travel for 1/4 sq km in 1hr
# each sq km is 18,000rs

# how long will the flight take?
# how much will it cost?
# where to bring down and put back up?


# how far can the drone get from the operator?
# how long can it be in flight?

# do the paths need to be circular?

# how to convert the path into "area" to use for the costing?


# flight number, km for each flight, time of each flight, number of buildings covered (with latlon), start point, end point, distance to cover by road to next start point

# fixed cost of time per building

## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona

fiona.drvsupport.supported_drivers["KML"] = "rw"

In [ ]:
from gridsample.utils import save_shapefiles
# from gridsample.mapping.plot import create_interactive_map

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
OUTPUT_DATA_DIR = DATA_DIR / "01_processed" / "Goverment Buildings"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

## Load data

In [ ]:
df = pd.read_csv(RAW_DATA_DIR / "gov_buildings" / "building_info_new.csv")

In [ ]:
df.columns

In [ ]:
df.rename(
    columns={
        "Latitude ": "lat",
        "Longitude": "lon",
    },
    inplace=True,
)

# drop any rows that have nulls in lat or lon
df = df.dropna(subset=["lat", "lon"])

In [ ]:
# convert to gdf
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326"
)

In [ ]:
gdf.plot(markersize=1)

In [ ]:
save_shapefiles(gdf, OUTPUT_DATA_DIR, "gov_buildings_new", formats=["parquet", "kml"])

In [ ]:
gdf.head()

## Clustering

In [ ]:
# cluster with kmeans
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=8, random_state=0)
gdf["cluster"] = [str(i) for i in kmeans.fit_predict(gdf[["lon", "lat"]])]
gdf.plot(column="cluster", markersize=1, legend=True)

In [ ]:
# from sklearn.cluster import HDBSCAN

# dbscan = HDBSCAN(n_jobs=-1)
# gdf["cluster"] = dbscan.fit_predict(gdf[["lon", "lat"]])
# gdf.plot(column="cluster", markersize=1, legend=True)

In [ ]:
gdf.dissolve("cluster").convex_hull.plot()

## Travelling Salesman

In [ ]:
gdf_projected = gdf.to_crs("EPSG:24378")

In [ ]:
gdf_projected.lat = gdf_projected.geometry.y
gdf_projected.lon = gdf_projected.geometry.x

In [ ]:
from scipy.spatial import distance_matrix

# Calculate the distance matrix
dist_matrix = distance_matrix(gdf_projected[["lon", "lat"]], gdf_projected[["lon", "lat"]])

In [ ]:
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

In [ ]:
# make list of latlon tuples
lonlat = [(lon, lat) for lon, lat in zip(gdf_projected.lon, gdf_projected.lat)]

In [ ]:
from scipy.spatial import distance_matrix
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

# Coordinates example
coordinates = lonlat

# Distance matrix
dist_matrix = distance_matrix(coordinates, coordinates)

def create_data_model():
    return {'distance_matrix': dist_matrix, 'num_locations': len(coordinates)}

data = create_data_model()
manager = pywrapcp.RoutingIndexManager(data['num_locations'], 1, 0)
routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data['distance_matrix'][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
search_parameters.time_limit.seconds = 30

solution = routing.SolveWithParameters(search_parameters)

In [ ]:
# Print the solution (ordered route)
if solution:
    index = routing.Start(0)
    route = []
    while not routing.IsEnd(index):
        route.append(manager.IndexToNode(index))
        index = solution.Value(routing.NextVar(index))
    route.append(manager.IndexToNode(index))
    print("Optimal Path:", route)

In [ ]:
# make a multi-line string
from shapely.geometry import LineString

In [ ]:
line_gdf = gpd.GeoDataFrame(
    data={"start_index": route[:-1], "end_index": route[1:]},
    geometry=[
        LineString(
            [
                gdf_projected.iloc[route[i]].geometry,
                gdf_projected.iloc[route[i + 1]].geometry,
            ]
        )
        for i in range(len(route) - 1)
    ],
)

In [ ]:
line_gdf

In [ ]:
# add start and end cluster names
line_gdf["start_cluster"] = gdf_projected.iloc[line_gdf.start_index].cluster.values
line_gdf["end_cluster"] = gdf_projected.iloc[line_gdf.end_index].cluster.values

In [ ]:
filtered_line_gdf = line_gdf[line_gdf["start_cluster"] == line_gdf["end_cluster"]]

In [ ]:
filtered_line_gdf.length.sum()

In [ ]:
fig, ax = plt.subplots(figsize=(15,15))
gdf_projected.plot(ax=ax, column="cluster", markersize=35)
# line_gdf.plot(ax=ax)
filtered_line_gdf.plot(ax=ax, color="black")

# add a 1km line to show scale on the plot
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
length = 10000
length_name = "10km"
ax.plot([xmax - length, xmax], [ymin, ymin], color="white", linewidth=5, linestyle="-")
ax.plot([xmax - length, xmax], [ymin, ymin], color="black", linewidth=2, linestyle="-")
ax.text(xmax - length/2, ymin + 400, length_name, fontsize=6, ha="center")

# axes off
ax.axis("off")

In [ ]:
flight_paths_gdf = filtered_line_gdf.dissolve("start_cluster").reset_index()[["start_cluster", "geometry"]]
flight_paths_gdf.rename(columns={"start_cluster": "cluster"}, inplace=True)
flight_paths_gdf.set_crs("EPSG:24378", inplace=True)

In [ ]:
# get count of buildings in each cluster
n_buildings_per_cluster = gdf_projected.groupby("cluster").size()
flight_paths_gdf["Number of Buildings"] = flight_paths_gdf["cluster"].map(n_buildings_per_cluster)

In [ ]:
flight_paths_gdf["Length (km)"] = round(flight_paths_gdf.length / 1000, 2)

In [ ]:
line_gdf[line_gdf["start_cluster"] != line_gdf["end_cluster"]].length

In [ ]:
save_shapefiles(
    flight_paths_gdf.to_crs("EPSG:4326"),
    OUTPUT_DATA_DIR,
    "flight_paths_grouped",
    formats=["parquet", "kml"],
)

In [ ]:
save_shapefiles(
    gdf.to_crs("EPSG:4326"),
    OUTPUT_DATA_DIR,
    "gov_buildings_grouped",
    formats=["parquet", "kml"],
)

In [ ]:
flight_path_buffered_gdf = flight_paths_gdf.copy()
flight_path_buffered_gdf["geometry"] = flight_path_buffered_gdf.buffer(50)

save_shapefiles(
    flight_path_buffered_gdf.to_crs("EPSG:4326"),
    OUTPUT_DATA_DIR,
    "flight_paths_grouped_buffered",
    formats=["parquet", "kml"],
)